[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/idhiatmojo/justrag/blob/main/notebooks/JustRAG_Colab_EndToEnd.ipynb)

# 🎓 Pipeline RAG Generatif Multi-Mapel Kurikulum Merdeka & Evaluasi Saintifik (S2)

Notebook interaktif **End-to-End** untuk menjalankan eksperimen riset pascasarjana (**JustRAG**): pembuatan **Modul Ajar**, **Paket Soal HOTS (Taksonomi Bloom C4–C6)**, **Ringkasan Materi**, serta **Studi Komparatif (*Ablation Study*)** berbasis dokumen buku ajar PDF yang diindeks secara dinamis.

### 🌟 Kontribusi & Fitur Utama yang Dapat Diuji di Notebook Ini:
1. **Isolasi Domain (*Metadata Pre-Filtering*)**: Mencegah kontaminasi silang (*cross-domain contamination*) antar mata pelajaran (Bahasa Inggris, Informatika, KKA, maupun mapel baru yang ditambahkan secara dinamis).
2. **Dynamic Incremental Indexing & Hybrid Retrieval**: Ekstraksi PDF + *Contextual Prefix Chunking* ke **ChromaDB (Dense Vector)** dan **BM25 (Sparse Lexical)** yang digabungkan menggunakan **Reciprocal Rank Fusion (RRF)** serta **Lexical-Semantic Cross-Reranker**.
3. **Manajemen Mapel & Akun Guru Dinamis (RBAC)**: Tambah mata pelajaran baru dan kelola hak akses guru secara terprogram maupun melalui UI.
4. **Evaluasi Kuantitatif 4 Varian Pipeline (*Ablation Study*)**: Mengukur *Hit Rate@4*, *MRR*, *Context Precision*, *Context Recall*, *Faithfulness*, *Answer Relevance*, dan *Bloom Compliance* lengkap dengan visualisasi grafik siap publikasi tesis/jurnal.
5. **Launcher Web UI Streamlit Penuh (Opsional)**: Menjalankan aplikasi web `app.py` lengkap dengan halaman Login & Panel Admin langsung dari Google Colab melalui *Cloudflare Quick Tunnel*.

---
> 💡 **Cara Pakai Cepat:** Klik **`Runtime` ➔ `Run all`** untuk menjalankan seluruh alur eksperimen otomatis dari instalasi hingga grafik evaluasi *Ablation Study* (bekerja baik menggunakan **API Key Cloud** maupun **100% Offline Mode** dengan FastEmbed Neural ONNX).

## 1️⃣ Persiapan Lingkungan & Instalasi Dependensi
Sel ini secara otomatis mendeteksi apakah notebook dijalankan di **Google Colab** atau **Jupyter Lokal**, mengkloning repositori `justrag` bila diperlukan, dan memasang seluruh pustaka yang dibutuhkan.

In [ ]:
#@title ⚙️ 1A. Clone Repository & Install Dependencies
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/idhiatmojo/justrag.git"

if IN_COLAB:
    PROJECT_ROOT = Path("/content/justrag")
    if not PROJECT_ROOT.exists():
        print(f"📥 Mengkloning repositori dari {REPO_URL} ...")
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
    else:
        print("🔄 Repositori sudah ada di /content/justrag, menarik pembaruan terbaru...")
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=False)

    os.chdir(PROJECT_ROOT)
    print("📦 Memasang dependensi Python (mohon tunggu ±30-60 detik pada eksekusi pertama)...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "matplotlib"],
        check=True,
    )
else:
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
    os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"✅ Direktori kerja aktif : {Path.cwd()}")
print(f"✅ Mode Google Colab     : {IN_COLAB}")

In [ ]:
#@title 💾 1B. (Opsional) Hubungkan Google Drive untuk Penyimpanan Permanen
#@markdown Centang `MOUNT_GOOGLE_DRIVE` jika Anda ingin database ChromaDB, indeks BM25, dan PDF yang diunggah tersimpan permanen di Google Drive Anda antar-sesi Colab.
MOUNT_GOOGLE_DRIVE = False #@param {type:"boolean"}
DRIVE_DATA_FOLDER = "/content/drive/MyDrive/justrag_data" #@param {type:"string"}

if IN_COLAB and MOUNT_GOOGLE_DRIVE:
    import shutil
    from google.colab import drive

    drive.mount("/content/drive")
    drive_path = Path(DRIVE_DATA_FOLDER)
    drive_path.mkdir(parents=True, exist_ok=True)

    local_data = PROJECT_ROOT / "data"
    if local_data.exists() and not local_data.is_symlink():
        for item in local_data.iterdir():
            dest = drive_path / item.name
            if not dest.exists():
                if item.is_dir():
                    shutil.copytree(item, dest)
                else:
                    shutil.copy2(item, dest)
        shutil.rmtree(local_data)

    if not local_data.exists():
        local_data.symlink_to(drive_path, target_is_directory=True)
    print(f"✅ Folder data berhasil dihubungkan ke Google Drive: {drive_path}")
else:
    print("ℹ️ Menggunakan penyimpanan lokal runtime sesi ini (data/).")

## 2️⃣ Konfigurasi Provider LLM, Embedding & Engine RAG
Anda dapat menyimpan API Key secara aman melalui menu **🔑 Secrets** di panel kiri Google Colab (dengan nama `GEMINI_API_KEY`, `OPENAI_API_KEY`, `XAI_API_KEY`, atau `CUSTOM_API_KEY`), atau mengisinya pada form di bawah.

> 💡 **Tanpa API Key?** Pilih `LLM_PROVIDER = "mock"` dan `EMBEDDING_PROVIDER = "local"`. Sistem tetap menjalankan **FastEmbed Neural Embedding (`all-MiniLM-L6-v2` 384-dim ONNX)** secara penuh untuk vektorisasi ChromaDB & BM25 tanpa biaya API!

In [ ]:
#@title 🤖 2. Inisialisasi LLM Client, ChromaDB, BM25 & Pipeline RAG
import os
import pandas as pd
try:
    from IPython.display import Markdown, display
except ImportError:
    display = print
    Markdown = lambda x: x

LLM_PROVIDER = "gemini" #@param ["gemini", "custom", "openai", "grok", "mock"]
MODEL_NAME = "gemini-2.5-flash" #@param {type:"string"}
API_KEY_MANUAL = "" #@param {type:"string"}
CUSTOM_BASE_URL = "https://api.juanrouter.com/v1" #@param {type:"string"}
EMBEDDING_PROVIDER = "local" #@param ["local", "gemini", "openai"]

def _read_colab_secret(secret_name: str) -> str:
    if not IN_COLAB:
        return ""
    try:
        from google.colab import userdata
        return userdata.get(secret_name) or ""
    except Exception:
        return ""

# Resolusi API Key otomatis: Input Form -> Colab Secrets -> Environment Variable
resolved_key = API_KEY_MANUAL.strip()
if not resolved_key:
    secret_map = {
        "gemini": ["GEMINI_API_KEY", "GOOGLE_API_KEY"],
        "openai": ["OPENAI_API_KEY"],
        "grok": ["XAI_API_KEY", "GROK_API_KEY"],
        "custom": ["CUSTOM_API_KEY", "JUANROUTER_API_KEY", "TAMANDATA_API_KEY"],
    }
    for s_key in secret_map.get(LLM_PROVIDER, []):
        val = _read_colab_secret(s_key) or os.getenv(s_key, "")
        if val:
            resolved_key = val.strip()
            print(f"🔐 API Key dimuat dari Secret/Env: {s_key}")
            break

emb_api_key = ""
if EMBEDDING_PROVIDER == "gemini":
    emb_api_key = _read_colab_secret("GEMINI_API_KEY") or os.getenv("GEMINI_API_KEY", "") or resolved_key
elif EMBEDDING_PROVIDER == "openai":
    emb_api_key = _read_colab_secret("OPENAI_API_KEY") or os.getenv("OPENAI_API_KEY", "") or resolved_key

from src.config import RAW_PDF_DIR, EVALUATION_DIR
from src.subject_manager import load_subjects, upsert_subject
from src.user_manager import load_users, create_user, get_accessible_subjects_for_user
from src.llm_factory import get_llm_client
from src.storage.vector_store import ChromaVectorStore
from src.storage.bm25_store import BM25SubjectStore
from src.ingestion.pipeline import DynamicIngestionPipeline
from src.retrieval.hybrid_retriever import HybridRetriever
from src.retrieval.reranker import LexicalSemanticReranker
from src.generation.generator import PedagogicalGenerator
from src.evaluation.evaluator import SystematicRAGEvaluator
from src.evaluation.synthetic_data import EvaluationDatasetManager

llm_client = get_llm_client(
    provider=LLM_PROVIDER,
    model_name=MODEL_NAME.strip() or None,
    api_key=resolved_key or None,
    base_url=CUSTOM_BASE_URL.strip() if LLM_PROVIDER == "custom" else None,
    embedding_provider=EMBEDDING_PROVIDER,
    embedding_api_key=emb_api_key or None,
)

vector_store = ChromaVectorStore()
bm25_store = BM25SubjectStore()
ingestion_pipeline = DynamicIngestionPipeline(vector_store=vector_store, bm25_store=bm25_store)
retriever = HybridRetriever(vector_store=vector_store, bm25_store=bm25_store, embedder=llm_client)
reranker = LexicalSemanticReranker()
generator = PedagogicalGenerator(llm_client=llm_client)

col_info = vector_store.get_active_collection_info(embedder=llm_client)
print(f"✅ LLM Client Aktif      : {llm_client.__class__.__name__} ({getattr(llm_client, 'model_name', '-')})")
print(f"✅ Koleksi ChromaDB Aktif: {col_info['name']} (Dimensi Vektor: {col_info['dimension']})")

## 3️⃣ Manajemen Mata Pelajaran (Mapel) & Akun Guru Dinamis
Secara bawaan proyek memuat 3 mata pelajaran Fase E Kelas X (**Bahasa Inggris**, **Informatika**, dan **KKA**). Anda dapat menambahkan mata pelajaran baru (misalnya *Matematika*, *Fisika*, *Sejarah*) langsung dari form berikut.

In [ ]:
#@title 📚 3. Lihat & Tambah Mata Pelajaran Baru Secara Dinamis
TAMBAH_MAPEL_BARU = False #@param {type:"boolean"}
NAMA_MAPEL_BARU = "Matematika (Kelas X)" #@param {type:"string"}
JENJANG_FASE = "Kelas X" #@param {type:"string"}
KODE_MAPEL_SLUG = "matematika" #@param {type:"string"}
DESKRIPSI_MAPEL = "Fase E SMA/SMK: Eksponen, Logaritma, Barisan dan Deret, Sistem Persamaan Linear, Trigonometri" #@param {type:"string"}

if TAMBAH_MAPEL_BARU:
    ok, msg, slug = upsert_subject(
        subject_key=KODE_MAPEL_SLUG,
        display_name=NAMA_MAPEL_BARU,
        description=DESKRIPSI_MAPEL,
        grade_level=JENJANG_FASE,
        is_new=True,
    )
    print(("✅ " if ok else "⚠️ ") + msg)

subjects = load_subjects()
df_subjects = pd.DataFrame(
    [
        {
            "Kode Mapel": k,
            "Nama Tampilan": v["display_name"],
            "Jenjang": v.get("grade_level", "Kelas X"),
            "Folder PDF": f"data/raw_pdf/{k}/",
            "Deskripsi Kurikulum": v["description"],
        }
        for k, v in subjects.items()
    ]
)
print(f"📋 Total Mata Pelajaran Terdaftar: {len(subjects)}")
display(df_subjects)

## 4️⃣ Ingesti Dokumen Kurikulum & Upload PDF Buku Ajar Sendiri
1. **Sel 4A** akan memastikan dokumen PDF contoh awal (Bahasa Inggris, Informatika, KKA) tersedia dan mengindeksnya ke **ChromaDB** + **BM25**.
2. **Sel 4B** memungkinkan Anda mengunggah file PDF modul/buku ajar milik Anda sendiri dari komputer langsung ke mata pelajaran yang dipilih di Google Colab.

In [ ]:
#@title 📥 4A. Generate Dokumen Contoh Awal & Indeks ke ChromaDB + BM25
from scripts.generate_sample_data import (
    generate_bahasa_inggris_sample,
    generate_informatika_sample,
    generate_kka_sample,
)

# Buat PDF contoh awal jika belum ada
generate_bahasa_inggris_sample()
generate_informatika_sample()
generate_kka_sample()

print("\n⚙️ Menjalankan ingesti dokumen ke ChromaDB (Dense) dan BM25 (Sparse)...")
ingest_results = ingestion_pipeline.ingest_all_in_folder(embedder=llm_client)
display(pd.DataFrame(ingest_results))

vec_stats = vector_store.get_stats_by_subject(embedder=llm_client)
bm25_stats = bm25_store.get_stats_by_subject()

summary_corpus = pd.DataFrame(
    [
        {
            "Kode Mapel": k,
            "Mata Pelajaran": v["display_name"],
            "Dense Vector Chunks (ChromaDB)": vec_stats.get(k, 0),
            "Sparse Lexical Chunks (BM25)": bm25_stats.get(k, 0),
        }
        for k, v in load_subjects().items()
    ]
)
print("📊 Ringkasan Indeks Korpus Saat Ini:")
display(summary_corpus)

In [ ]:
#@title 📄 4B. (Opsional) Upload File PDF Materi Sendiri dari Komputer
#@markdown Ubah `UPLOAD_PDF_SEKARANG = True` lalu jalankan sel ini untuk memilih file `.pdf` dari komputer Anda.
UPLOAD_PDF_SEKARANG = False #@param {type:"boolean"}
TARGET_KODE_MAPEL = "informatika" #@param {type:"string"}

if UPLOAD_PDF_SEKARANG:
    active_subs = load_subjects()
    target_sub = TARGET_KODE_MAPEL.lower().strip()
    if target_sub not in active_subs:
        raise ValueError(f"Kode mapel '{target_sub}' belum terdaftar. Pilihan: {list(active_subs.keys())}")

    if not IN_COLAB:
        print("ℹ️ Fitur dialog upload interaktif khusus Google Colab. Di lokal, salin PDF langsung ke:", RAW_PDF_DIR / target_sub)
    else:
        from google.colab import files

        print(f"📤 Silakan pilih satu atau beberapa file PDF untuk mapel '{active_subs[target_sub]['display_name']}':")
        uploaded = files.upload()
        target_dir = RAW_PDF_DIR / target_sub
        target_dir.mkdir(parents=True, exist_ok=True)

        for fname, content in uploaded.items():
            clean_name = Path(fname).name
            if not clean_name.lower().endswith(".pdf"):
                print(f"⚠️ Melewati '{clean_name}' (bukan file PDF).")
                continue
            save_path = target_dir / clean_name
            with open(save_path, "wb") as f:
                f.write(content)
            res = ingestion_pipeline.ingest_single_pdf(
                pdf_path=save_path,
                subject=target_sub,
                embedder=llm_client,
                grade_level=active_subs[target_sub].get("grade_level", "Kelas X"),
            )
            print(f"✅ Berhasil mengindeks '{clean_name}': {res['pages']} halaman -> {res['chunks']} chunks.")
else:
    print("ℹ️ Lewati upload manual (set UPLOAD_PDF_SEKARANG = True jika ingin mengunggah PDF sendiri).")

## 5️⃣ Eksplorasi Hybrid Retrieval (Dense + BM25 RRF) & Cross-Reranking
Bagian ini memperlihatkan cara kerja mesin pencari hibrida dan membuktikan bagaimana `USE_SUBJECT_FILTER` mengisolasi domain materi agar tidak tercampur antar mata pelajaran.

In [ ]:
#@title 🔍 5. Uji Pencarian Konteks & Perbandingan Skor Reranking
KODE_MAPEL_UJI = "informatika" #@param {type:"string"}
PERTANYAAN_UJI = "Jelaskan empat pilar berpikir komputasional serta perbedaan topologi star dan bus" #@param {type:"string"}
MODE_RETRIEVAL = "hybrid" #@param ["hybrid", "dense_only", "bm25_only"]
ISOLASI_FILTER_MAPEL = True #@param {type:"boolean"}
AKTIFKAN_RERANKER = True #@param {type:"boolean"}

candidate_k = 6 if AKTIFKAN_RERANKER else 4
raw_hits = retriever.retrieve(
    query=PERTANYAAN_UJI,
    subject=KODE_MAPEL_UJI,
    mode=MODE_RETRIEVAL,
    use_subject_filter=ISOLASI_FILTER_MAPEL,
    top_k=candidate_k,
)

final_hits = (
    reranker.rerank(query=PERTANYAAN_UJI, candidates=raw_hits, top_k=4)
    if (AKTIFKAN_RERANKER and raw_hits)
    else raw_hits[:4]
)

rows_hits = []
for rank, ch in enumerate(final_hits, 1):
    meta = ch.get("metadata", {})
    rows_hits.append(
        {
            "Rank": rank,
            "Mapel": meta.get("subject", "-"),
            "File Sumber": meta.get("source_file", "-"),
            "Hal.": meta.get("page_number", "-"),
            "Tipe": ch.get("retrieval_type", "-"),
            "Skor Awal/RRF": round(ch.get("rrf_score", ch.get("score", 0.0)), 4),
            "Skor Rerank": ch.get("rerank_score", "-"),
            "Cuplikan Teks": ch.get("text", "")[:180].replace("\n", " ") + "...",
        }
    )

print(f"🔎 Ditemukan {len(final_hits)} chunk konteks teratas:")
display(pd.DataFrame(rows_hits))

## 6️⃣ Generasi Bahan Ajar & Asesmen HOTS Kurikulum Merdeka
Mendukung 4 mode pedagogis terstruktur:
- `modul_ajar` — Rencana/Modul Ajar lengkap (CP/ATP, Pertanyaan Pemantik, Kegiatan Inti, Asesmen).
- `soal_hots` — Paket Soal Pilihan Ganda & Studi Kasus level kognitif Bloom **C4–C6** beserta stimulus dan pembahasan.
- `ringkasan` — Rangkuman konsep utama, analogi siswa, dan glosarium.
- `qa_chat` — Tanya jawab terpandu berbasis kutipan halaman buku ajar.

In [ ]:
#@title ✨ 6. Jalankan Generasi Konten Pedagogis Berbasis RAG
MAPEL_GENERASI = "kka" #@param {type:"string"}
MODE_PEDAGOGIS = "soal_hots" #@param ["modul_ajar", "soal_hots", "ringkasan", "qa_chat"]
STRATEGI_SEARCH = "hybrid" #@param ["hybrid", "dense_only", "bm25_only"]
TEMPERATURE = 0.3 #@param {type:"slider", min:0.0, max:1.0, step:0.1}
INSTRUKSI_TOPIK = "Buat soal HOTS analisis perbandingan Supervised vs Unsupervised Learning serta mitigasi bias algoritma AI" #@param {type:"string"}

raw_ctx = retriever.retrieve(
    query=INSTRUKSI_TOPIK,
    subject=MAPEL_GENERASI,
    mode=STRATEGI_SEARCH,
    use_subject_filter=True,
    top_k=6,
)
ranked_ctx = reranker.rerank(query=INSTRUKSI_TOPIK, candidates=raw_ctx, top_k=4)

hasil = generator.generate(
    query=INSTRUKSI_TOPIK,
    subject=MAPEL_GENERASI,
    retrieved_chunks=ranked_ctx,
    task_type=MODE_PEDAGOGIS,
    temperature=TEMPERATURE,
)

display(Markdown("---"))
display(Markdown(hasil["response"]))
display(Markdown("---"))
print("📚 Sitasi Dokumen Grounding yang Digunakan:")
display(pd.DataFrame(hasil["sources"]))

## 7️⃣ Eksperimen Evaluasi Saintifik & Ablation Study (Standar Riset S2)
Menguji **4 konfigurasi arsitektur RAG** secara sistematis terhadap *Golden Testset*:
1. **1. Baseline (Naive Dense, No Filter)** — Mengukur dampak tanpa isolasi metadata mapel.
2. **2. Naive + Mapel Filter** — Mengukur kontribusi *domain metadata pre-filtering*.
3. **3. Hybrid (Dense+BM25) + Filter** — Mengukur kontribusi fusi pencarian vektor + leksikal (RRF).
4. **4. Proposed (Hybrid+Filter+Reranker)** — Arsitektur lengkap yang diusulkan.

In [ ]:
#@title 📊 7A. Jalankan Benchmark Ablation Study & Visualisasi Grafik Publikasi
import numpy as np

CAKUPAN_EVALUASI = "SEMUA_MAPEL" #@param {type:"string"}
#@markdown *Isi `SEMUA_MAPEL` untuk pengujian lintas-domain penuh, atau ketik kode mapel tertentu (mis. `informatika`, `bahasa_inggris`, `kka`).*

target_eval_sub = None if CAKUPAN_EVALUASI.strip().upper() in ("", "SEMUA_MAPEL", "ALL") else CAKUPAN_EVALUASI.strip().lower()

evaluator = SystematicRAGEvaluator(retriever=retriever, generator=generator)
df_ablation = evaluator.run_ablation_benchmark(subject=target_eval_sub)

print("✅ Hasil Matriks Kuantitatif Ablation Study:")
display(df_ablation)

csv_path = EVALUATION_DIR / "ablation_benchmark_results.csv"
chart_path = EVALUATION_DIR / "ablation_benchmark_chart.png"

try:
    import matplotlib.pyplot as plt

    # Visualisasi Grafik Berstandar Publikasi Tesis / Jurnal
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), dpi=140)
    configs_short = ["1. Baseline\n(No Filter)", "2. Dense +\nFilter Mapel", "3. Hybrid RRF\n+ Filter", "4. Proposed\n(Full Pipeline)"]
    if len(df_ablation) != 4:
        configs_short = df_ablation["config"].tolist()

    x = np.arange(len(df_ablation))
    width = 0.2

    # Subplot 1: Metrik Retrieval
    ret_metrics = [
        ("hit_rate@4", "Hit Rate@4", "#2563eb"),
        ("mrr", "MRR", "#0ea5e9"),
        ("context_precision", "Context Precision", "#10b981"),
        ("context_recall", "Context Recall", "#f59e0b"),
    ]
    for idx, (col, label, color) in enumerate(ret_metrics):
        axes[0].bar(x + (idx - 1.5) * width, df_ablation[col], width, label=label, color=color, alpha=0.9)

    axes[0].set_title("Perbandingan Performa Retrieval (Ablation Study)", fontsize=12, fontweight="bold")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(configs_short, fontsize=9)
    axes[0].set_ylim(0, 1.12)
    axes[0].set_ylabel("Skor (0.0 – 1.0)")
    axes[0].grid(axis="y", linestyle="--", alpha=0.4)
    axes[0].legend(loc="lower right", fontsize=8.5)

    # Subplot 2: Metrik Generasi & Kepatuhan Taksonomi Bloom
    gen_metrics = [
        ("faithfulness", "Faithfulness (Grounded)", "#7c3aed"),
        ("answer_relevance", "Answer Relevance", "#ec4899"),
        ("bloom_compliance", "Bloom Alignment", "#14b8a6"),
    ]
    width_g = 0.24
    for idx, (col, label, color) in enumerate(gen_metrics):
        axes[1].bar(x + (idx - 1.0) * width_g, df_ablation[col], width_g, label=label, color=color, alpha=0.9)

    axes[1].set_title("Kualitas Generasi & Kesesuaian Pedagogis", fontsize=12, fontweight="bold")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(configs_short, fontsize=9)
    axes[1].set_ylim(0, 1.12)
    axes[1].set_ylabel("Skor (0.0 – 1.0)")
    axes[1].grid(axis="y", linestyle="--", alpha=0.4)
    axes[1].legend(loc="lower right", fontsize=8.5)

    plt.tight_layout()
    plt.savefig(chart_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"🖼️ Grafik PNG tersimpan di: {chart_path}")
except ImportError:
    print("ℹ️ Pustaka matplotlib belum terpasang di environment lokal ini (otomatis terpasang di Google Colab).")

print(f"💾 Tabel CSV tersimpan di : {csv_path}")

In [ ]:
#@title ⬇️ 7B. (Opsional) Unduh File CSV & Grafik PNG ke Komputer Anda
UNDUH_HASIL_EVALUASI = False #@param {type:"boolean"}

if IN_COLAB and UNDUH_HASIL_EVALUASI:
    from google.colab import files

    csv_file = EVALUATION_DIR / "ablation_benchmark_results.csv"
    png_file = EVALUATION_DIR / "ablation_benchmark_chart.png"
    if csv_file.exists():
        files.download(str(csv_file))
    if png_file.exists():
        files.download(str(png_file))
else:
    print("ℹ️ Set UNDUH_HASIL_EVALUASI = True di Google Colab untuk mengunduh laporan CSV & grafik PNG.")

## 8️⃣ (Opsional) Jalankan Full Web UI Streamlit (`app.py`) dari Google Colab
Ingin mencoba antarmuka web lengkap (termasuk **Halaman Login**, **Upload PDF Interaktif**, dan **Panel Admin Manajemen Mapel & Guru**) langsung dari Google Colab?

1. Centang `JALANKAN_STREAMLIT_UI = True` pada form di bawah lalu jalankan selnya.
2. Klik tautan publik **`https://....trycloudflare.com`** yang muncul pada output.
3. Login menggunakan akun bawaan:
   - **Admin**: `admin` / `admin123` *(Akses penuh + Tab Manajemen Mapel & Guru + Tab Evaluasi)*
   - **Guru**: `guru` / `guru123` *(Akses pembuatan bahan ajar & upload PDF mapel)*

In [ ]:
#@title 🌐 8. Jalankan Server Streamlit + Cloudflare Quick Tunnel
JALANKAN_STREAMLIT_UI = False #@param {type:"boolean"}

if JALANKAN_STREAMLIT_UI:
    import re
    import time
    import urllib.request
    import subprocess

    cloudflared_bin = Path("/tmp/cloudflared")
    if not cloudflared_bin.exists():
        print("⬇️ Mengunduh Cloudflare Tunnel binary...")
        urllib.request.urlretrieve(
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            str(cloudflared_bin),
        )
        cloudflared_bin.chmod(0o755)

    # Teruskan API key sesi notebook ke environment proses Streamlit bila tersedia
    env_vars = os.environ.copy()
    if resolved_key and LLM_PROVIDER == "gemini":
        env_vars["GEMINI_API_KEY"] = resolved_key
    elif resolved_key and LLM_PROVIDER == "openai":
        env_vars["OPENAI_API_KEY"] = resolved_key

    print("🚀 Menjalankan server Streamlit di background (port 8501)...")
    st_proc = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "streamlit",
            "run",
            "app.py",
            "--server.port=8501",
            "--server.address=0.0.0.0",
            "--server.headless=true",
            "--server.enableCORS=false",
            "--server.enableXsrfProtection=false",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        env=env_vars,
    )
    time.sleep(3)

    print("🔗 Membuka terowongan HTTPS publik (Cloudflare Quick Tunnel)...")
    cf_proc = subprocess.Popen(
        [str(cloudflared_bin), "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    public_url = None
    start_wait = time.time()
    while time.time() - start_wait < 30:
        line = cf_proc.stdout.readline()
        if not line:
            break
        match = re.search(r"(https://[a-zA-Z0-9-]+\.trycloudflare\.com)", line)
        if match:
            public_url = match.group(1)
            break

    if public_url:
        print("\n" + "=" * 72)
        print(f"✅ WEB UI STREAMLIT AKTIF! Klik link berikut:\n👉  {public_url}")
        print("=" * 72)
        print("🔐 Akun Demo -> Admin: admin / admin123 | Guru: guru / guru123")
        print("⏹️ Biarkan sel ini berjalan selama menggunakan UI. Klik tombol Stop untuk mematikan server.")
        try:
            st_proc.wait()
        except KeyboardInterrupt:
            print("\n🛑 Menghentikan server Streamlit & tunnel...")
        finally:
            st_proc.terminate()
            cf_proc.terminate()
    else:
        st_proc.terminate()
        cf_proc.terminate()
        print("❌ Gagal mendapatkan URL publik Cloudflare Tunnel. Silakan jalankan ulang sel ini.")
else:
    print("ℹ️ Server Web UI tidak dijalankan (set JALANKAN_STREAMLIT_UI = True bila ingin membuka Web UI Streamlit).")